# Advanced Agentic RAG System

An Agentic Retrieval-Augmented Generation (RAG) system
built using Python, Groq LLM, and ChromaDB.

### Key Features
- Document chunking
- LLM-based chunk enrichment
- Vector search using ChromaDB
- Reciprocal Rank Fusion (RRF)
- Tool-based retrieval
- Agentic reasoning and response generation

### Dataset
A fictional employee company policy handbook.

## Project Workflow

1. Load the company policy document.
2. Split the document into smaller chunks.
3. Enrich chunks using an LLM.
4. Store original and enriched chunks in ChromaDB.
5. Retrieve relevant chunks from both collections.
6. Combine retrieval results using Reciprocal Rank Fusion (RRF).
7. Expose retrieval through a search tool.
8. Let the AI agent call the search tool and generate an answer.

## Technology Stack

- Python
- Groq API – LLM inference
- ChromaDB – Vector database
- Reciprocal Rank Fusion – Hybrid retrieval
- Google Colab – Development environment

In [ ]:
!pip install -q groq chromadb

In [ ]:
import os
import json
import chromadb

from groq import Groq

print("All packages imported successfully!")

In [ ]:
from google.colab import userdata

In [ ]:
groqapikey = userdata.get("Groq")

if not groqapikey:
    raise ValueError("Groq API key not found. Please add it to Colab Secrets.")

client = Groq(api_key=groqapikey)

print("Groq client connected successfully!")

Test LLM Connection

In [ ]:
response=client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role":"user",
            "content": "Explain what is RAG in one sentence."
        }
    ]

)
print(response.choices[0].message.content)

In [ ]:
filepath="/content/sample_company_policy_for_rag.txt"
with open(filepath, "r", encoding="utf-8") as f:
  policy_document=f.read()
print("Document loaded successfully")


In [ ]:
print("Document_length:", len(policy_document))
print("\nFirst 500 characters:\n")
print(policy_document[:500])

Document Chunking

In [ ]:
def chunk_document(text, source_name):
  paragraph=text.split("\n\n")

  chunks=[]
  for para in paragraph:
    para=para.strip()
    if len(para)< 50:
      continue
    if para.startswith("===="):
      continue
    chunks.append({
        "text":para,
        "source":source_name
    })
  return chunks


In [ ]:
chunks=chunk_document(policy_document,"sample_company_policy_for_rag.txt")

print("Total chunks created:", len(chunks))
print("\nFirst chunk:\n")
print(json.dumps(chunks[0], indent=2))

Enrichment Function


In [ ]:

def enrich_chunk(chunk):

    prompt = f"""
    Analyze the following company policy text.

    Return ONLY valid JSON with these four fields:
    1. summary
    2. keywords
    3. topic
    4. original_content

    Text:
    {chunk}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    result = response.choices[0].message.content.strip()

    # Remove Markdown code block formatting
    if result.startswith("```json"):
        result = result[len("```json"):]

    elif result.startswith("```"):
        result = result[len("```"):]

    if result.endswith("```"):
        result = result[:-3]

    result = result.strip()

    # Convert JSON string into Python dictionary
    enriched_data = json.loads(result)

    return enriched_data

In [ ]:
enriched_chunks = []

for i, chunk in enumerate(chunks):

    print(f"Enriching chunk {i+1}/{len(chunks)}")

    enriched_data = enrich_chunk(chunk["text"])

    enriched_chunks.append({
        "text": chunk["text"],
        "source": chunk["source"],
        "summary": enriched_data["summary"],
        "keywords": enriched_data["keywords"],
        "topic": enriched_data["topic"]
    })

print("\nTotal enriched chunks:", len(enriched_chunks))

Create Function for Formating Text

In [ ]:
def create_enriched_text(chunk):

    enriched_text = f"""
    Topic: {chunk['topic']}

    Summary: {chunk['summary']}

    Keywords: {', '.join(chunk['keywords'])}

    Original Content: {chunk['text']}
    """

    return enriched_text.strip()

In [ ]:
sample_enriched_text = create_enriched_text(enriched_chunks[0])

print(sample_enriched_text)

In [ ]:
enriched_texts = []

for chunk in enriched_chunks:

    text = create_enriched_text(chunk)

    enriched_texts.append(text)

print("Total enriched texts:", len(enriched_texts))

In [ ]:
chroma_client = chromadb.Client()

print("ChromaDB initialized successfully!")

In [ ]:
original_collection = chroma_client.get_or_create_collection(
    name="company_docs_original"
)

enriched_collection = chroma_client.get_or_create_collection(
    name="company_docs_enriched"
)

print("Both collections created successfully!")

Store Original Chunks in ChromaDB

In [ ]:
original_collection.add(
    ids=[f"original_{i}" for i in range(len(chunks))],

    documents=[chunk["text"] for chunk in chunks],

    metadatas=[
        {"source": chunk["source"]}
        for chunk in chunks
    ]
)

print("Original chunks stored successfully!")
print("Total records:", original_collection.count())

Store Enriched Chunks in ChromaDB

In [ ]:
enriched_collection.upsert(
    ids=[f"enriched_{i}" for i in range(len(enriched_chunks))],
    documents=enriched_texts,
    metadatas=[
        {
        "source":chunk["source"],
        "topic":chunk["topic"],
        }
        for chunk in enriched_chunks
    ]
)
print("Enriched chunks stored successfully!")
print("Total records:", enriched_collection.count())

In [ ]:
results=original_collection.query(
    query_texts=["Who is responsible for updating company policies?"],
    n_results=3

)
print(results["documents"])

In [ ]:
for i, doc in enumerate(results["documents"][0]):

    print(f"\nResult {i+1}:")
    print(doc)

Test Enriched Retrieval

In [ ]:
enriched_results = enriched_collection.query(
    query_texts=["Who is responsible for updating company policies?"],
    n_results=3
)

print(enriched_results["documents"])

In [ ]:
for i, doc in enumerate(enriched_results["documents"][0]):

    print(f"\nResult {i+1}:")
    print(doc)

Compare Original vs. Enriched Retrieval Distances

In [ ]:
original_results = original_collection.query(
    query_texts=["Who is responsible for updating company policies?"],
    n_results=3,
    include=["documents", "distances"]
)

In [ ]:
enriched_results = enriched_collection.query(
    query_texts=["Who is responsible for updating company policies?"],
    n_results=3,
    include=["documents", "distances"]
)

In [ ]:
print("ORIGINAL RETRIEVAL")

for i, distance in enumerate(original_results["distances"][0]):
    print(f"Result {i+1}: Distance = {distance:.4f}")


print("\nENRICHED RETRIEVAL")

for i, distance in enumerate(enriched_results["distances"][0]):
    print(f"Result {i+1}: Distance = {distance:.4f}")

RRF Function(with deduplication Internally)

In [ ]:
def reciprocal_rank_fusion(original_results, enriched_results, k=60):

    combined = {}

    # Process original retrieval results
    for rank, (doc_id, doc) in enumerate(
        zip(original_results["ids"][0],
            original_results["documents"][0]), start=1
    ):

        chunk_id = doc_id.split("_")[1]

        if chunk_id not in combined:
            combined[chunk_id] = {
                "chunk_id": chunk_id,
                "original_text": None,
                "enriched_text": None,
                "original_rank": None,
                "enriched_rank": None,
                "rrf_score": 0
            }

        combined[chunk_id]["original_text"] = doc
        combined[chunk_id]["original_rank"] = rank

        combined[chunk_id]["rrf_score"] += 1 / (k + rank)

    # Process enriched retrieval results
    for rank, (doc_id, doc) in enumerate(
        zip(enriched_results["ids"][0],
            enriched_results["documents"][0]), start=1
    ):

        chunk_id = doc_id.split("_")[1]

        if chunk_id not in combined:
            combined[chunk_id] = {
                "chunk_id": chunk_id,
                "original_text": None,
                "enriched_text": None,
                "original_rank": None,
                "enriched_rank": None,
                "rrf_score": 0
            }

        combined[chunk_id]["enriched_text"] = doc
        combined[chunk_id]["enriched_rank"] = rank

        combined[chunk_id]["rrf_score"] += 1 / (k + rank)

    # Sort chunks by RRF score (highest first)
    ranked_results = sorted(
        combined.values(),
        key=lambda x: x["rrf_score"],
        reverse=True
    )

    return ranked_results

Testing RRF function

In [ ]:
ranked_results = reciprocal_rank_fusion(
    original_results,
    enriched_results
)

print("Total unique chunks:", len(ranked_results))

for result in ranked_results:

    print("\nChunk ID:", result["chunk_id"])
    print("Original Rank:", result["original_rank"])
    print("Enriched Rank:", result["enriched_rank"])
    print("RRF Score:", round(result["rrf_score"], 6))

Integrating RRF into combined retrieval

In [ ]:
def combined_retrieval(question, n_results=3):

    # Retrieve from original collection
    original_results = original_collection.query(
        query_texts=[question],
        n_results=n_results
    )

    # Retrieve from enriched collection
    enriched_results = enriched_collection.query(
        query_texts=[question],
        n_results=n_results
    )

    # Combine and rank using RRF
    ranked_results = reciprocal_rank_fusion(
        original_results,
        enriched_results
    )

    return ranked_results

Testing

In [ ]:
question = "Who is responsible for updating company policies?"

results = combined_retrieval(question, n_results=3)

for result in results:

    print("\nChunk ID:", result["chunk_id"])
    print("RRF Score:", round(result["rrf_score"], 6))
    print("Original Rank:", result["original_rank"])
    print("Enriched Rank:", result["enriched_rank"])

Creating Search Tool

In [ ]:

def search_company_policy(query):

    # Use combined retrieval (Original + Enriched + RRF)
    results = combined_retrieval(query, n_results=8)

    search_results = []

    for result in results:

        # Prefer original policy text
        content = result["original_text"]

        # If original text is unavailable, use enriched text
        if content is None:
            content = result["enriched_text"]

        search_results.append({
            "chunk_id": result["chunk_id"],
            "content": content,
            "rrf_score": result["rrf_score"]
        })

    return search_results


Testing the tool

In [ ]:
tool_result = search_company_policy(
    "How many paid leaves a full time employee receive?"
)

for result in tool_result:
    print("CHUNK ID:", result["chunk_id"])
    print("CONTENT:", result["content"])
    print("RRF SCORE:", result["rrf_score"])
    print("-" * 80)

Create the tool schema

In [ ]:
rag_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_company_policy",
            "description": "Search the company policy document for relevant information about employee policies, rules, security, AI usage, privacy, and company procedures.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The question or search query to use when searching the company policy document."
                    }
                },
                "required": ["query"]
            }
        }
    }
]

Testing Tool Schema

In [ ]:
print(json.dumps(rag_tools, indent=4))

In [ ]:

conversation_history = []

def rag_agent(question, max_iterations=5):

    global conversation_history

    system_prompt = """
    You are a company policy assistant.

IMPORTANT ANSWERING RULES:

1. Always search the company policy document before answering.

2. WHEN THE ANSWER IS IN THE POLICY:
   - Use the exact original wording from the retrieved policy.
   - Do not paraphrase, rewrite, summarize, or change sentences.
   - Do not add information that is not present in the retrieved text.
   - Do not repeat the same information twice.
   - Preserve the original bullet points and terminology.
   - Include the source chunk ID.

3. WHEN THE ANSWER IS NOT IN THE POLICY:
   - Clearly state that the information is not specified
     in the provided policy.
   - You may then provide general knowledge if appropriate.
   - Clearly label general knowledge separately.
   - Never present general knowledge as company policy.
   - Never invent company-specific facts.

4. Do not output raw JSON or repeat the tool response.

5. Keep the answer clear and well-formatted.
    """

    messages = [
        {"role": "system", "content": system_prompt}
    ] + conversation_history + [
        {"role": "user", "content": question}
    ]

    for iteration in range(max_iterations):

        # Call LLM with tools available on every iteration
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages,
            tools=rag_tools,
            tool_choice="auto",
            temperature=0
        )

        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        # If no tool call, return the final answer
        if not assistant_message.tool_calls:

            answer = assistant_message.content

            conversation_history.extend([
                {"role": "user", "content": question},
                {"role": "assistant", "content": answer}
            ])

            print(answer)
            return answer

        # Execute all requested tool calls
        for tool_call in assistant_message.tool_calls:

            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            if tool_name == "search_company_policy":
                tool_result = search_company_policy(**arguments)
            else:
                raise ValueError(f"Unknown tool: {tool_name}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_result)
            })

    print("Maximum tool-call iterations reached.")
    return None


def reset_rag_conversation():
    global conversation_history
    conversation_history = []
    print("Conversation history cleared!")


In [ ]:
rag_agent("What is the standard work schedule?")

In [ ]:
rag_agent("What is the age of cricketer Dhoni?")

In [ ]:
rag_agent("Who is abdulkalam?")

In [ ]:
 rag_agent("What is the WORKING HOURS AND ATTENDANCE")

In [ ]:
rag_agent("What is the company's sick leave policy?")

In [ ]:
rag_agent("What is the company's maternity leave duration?")

In [ ]:
rag_agent("What is the capital of Japan??")